# Actividad: Agente Reactivo Simple

En esta actividad implementaremos y evaluaremos un **agente reactivo simple**.

Un agente reactivo simple selecciona una acción únicamente a partir del estado actual:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

donde:

- $\mathcal{S}$ es el conjunto de estados observables.
- $\mathcal{A}$ es el conjunto de acciones.
- El agente no conserva memoria de estados anteriores.
- El agente no realiza planificación.

El objetivo de la actividad es:

1. Implementar un agente reactivo mediante una política tabular.
2. Implementar un agente aleatorio como referencia.
3. Ejecutar episodios de interacción agente-entorno.
4. Comparar experimentalmente el desempeño de ambos agentes.

In [8]:
# Importar bibliotecas necesarias
import random

## Entorno: GridWorld

Utilizaremos un entorno discreto representado mediante una cuadrícula de tamaño `size × size`.

### Componentes del entorno

- **Estado:** posición actual del agente `(x, y)`.
- **Estado inicial:** definido por `start`.
- **Estado meta:** definido por `goal`.
- **Acciones posibles:** `UP`, `DOWN`, `LEFT`, `RIGHT`.

El espacio de estados puede representarse como:

$\mathcal{S} = \{(x,y) \mid 0 \leq x,y < size\}$

y el conjunto de acciones como:

$\mathcal{A} = \{UP, DOWN, LEFT, RIGHT\}$

El entorno es determinista: una acción válida produce siempre el movimiento correspondiente.

La función de transición puede representarse como:

$\mathcal{T} : \mathcal{S} \times \mathcal{A} \rightarrow \mathcal{S}$

### Métodos principales

- `reset()` reinicia el entorno al estado inicial.
- `perceive()` devuelve las acciones válidas desde el estado actual.
- `step(action)` ejecuta una acción y actualiza el estado.
- `is_done()` verifica si el agente alcanzó la meta.

In [3]:
# -----------------------------
# Entorno
# -----------------------------
class GridWorld:
    def __init__(self, size=3, start=(0, 0), goal=(2, 2)):
        self.size = size
        self.start = start
        self.goal = goal
        self.agent_pos = start

    def reset(self):
        self.agent_pos = self.start

    def perceive(self):
        actions = []
        size = self.size
        x,y = self.agent_pos
        if x < size-1:
            actions.append("RIGHT")
        if x > 0:
            actions.append("LEFT")
        if y < size-1:
            actions.append("UP")
        if y > 0:
            actions.append("DOWN")
            
        return actions

    def step(self,action):
        x,y = self.agent_pos
        if action == "UP":
            y = y + 1
        elif action == "DOWN":
            y = y - 1
        elif action == "LEFT":
            x = x - 1
        elif action == "RIGHT":
            x = x + 1
        self.agent_pos = x,y

    def is_done(self):
        if self.agent_pos == self.goal:
            return True
        else:
            return False



## Ejecución de un episodio

Para evaluar un agente necesitamos ejecutar su interacción con el entorno durante varios pasos.

En cada instante:

1. El agente observa el estado actual.
2. Selecciona una acción.
3. El entorno ejecuta la acción.
4. Se actualiza el estado.
5. Se verifica si se alcanzó la meta.

Un episodio termina cuando:

- el agente alcanza la meta, o
- se alcanza un número máximo de pasos.

El límite de pasos evita ejecuciones indefinidas cuando una política no logra alcanzar la meta.

La función `run_episode()` devuelve:

- `success`: indica si el agente alcanzó la meta.
- `steps`: número de acciones ejecutadas.
- `trajectory`: secuencia de estados visitados.

In [4]:
def run_episode(env,agent,max_steps):
    env.reset()
    steps = 0
    trajectory = [env.start]
    while not env.is_done() and steps < max_steps:
        agent.act()
        steps = steps + 1
        trajectory.append(env.agent_pos)
    if steps <= max_steps and env.is_done():
        return True, steps, trajectory
    else:
        return False, steps, trajectory

## Agente Reactivo Simple

Representaremos la política del agente mediante un diccionario.

Un agente reactivo simple implementa una política:

$\pi : \mathcal{S} \rightarrow \mathcal{A}$

de modo que a cada estado $s$ se le asigna directamente una acción $a$:

$\pi(s) = a$

El agente:

- observa únicamente el estado actual;
- consulta la acción asociada con ese estado;
- no conserva memoria;
- no realiza búsqueda;
- no analiza estados futuros.

La política contiene, por tanto, conocimiento previamente especificado sobre cómo debe actuar el agente en cada estado.

In [5]:
# -----------------------------
# Agente Reactivo Simple
# -----------------------------

class SimpleReactiveAgent:

    def __init__(self, env, policy):
        self.env = env
        self.policy = policy
        
    def act(self):
        done = self.env.is_done()
        if done:
            return 0
        else:
            x,y = self.env.agent_pos
            actions = self.env.perceive()
            action = self.policy.get((x,y))
            if action in actions:
                self.env.step(action)    
        

## Política tabular

Definiremos ahora una política para un entorno de tamaño `3 × 3`, con:

- estado inicial `(0, 0)`;
- meta `(2, 2)`.

Cada entrada del diccionario representa una regla de la forma:

`estado → acción`

Por ejemplo:

`(0, 0) → RIGHT`

significa que cuando el agente se encuentre en el estado `(0, 0)`, ejecutará la acción `RIGHT`.

In [6]:
# Política determinista π : S → A
policy = {
    (0, 0): "RIGHT",
    (0, 1): "RIGHT",
    (0, 2): "RIGHT",

    (1, 0): "RIGHT",
    (1, 1): "RIGHT",
    (1, 2): "RIGHT",

    (2, 0): "UP",
    (2, 1): "UP",

    (2, 2): None
}

## Prueba del agente reactivo

Ejecutaremos un episodio y observaremos:

- si el agente alcanza la meta;
- cuántos pasos utiliza;
- qué trayectoria sigue.

In [7]:
env_reactive = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

reactive_agent = SimpleReactiveAgent(
    env_reactive,
    policy
)

success, steps, trajectory = run_episode(
    env_reactive,
    reactive_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: True
Pasos: 4
Trayectoria: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2)]


## Análisis de la trayectoria

Desde `(0,0)` hasta `(2,2)`, la distancia mínima en una cuadrícula sin obstáculos es:

$d^* = |x_g-x_0| + |y_g-y_0|$

En este caso:

$d^* = |2-0| + |2-0| = 4$

Compara este valor con el número de pasos utilizado por el agente.

Observa que el agente puede seguir una trayectoria óptima aun cuando **no realiza búsqueda ni planificación**.

La trayectoria óptima está implícitamente codificada en la política.

## Agente Aleatorio

Utilizaremos ahora un agente aleatorio como línea base de comparación.

Este agente:

1. obtiene las acciones válidas en el estado actual;
2. selecciona una de ellas al azar.

Si existen $k$ acciones válidas, cada acción tiene probabilidad:

$P(a \mid s) = \frac{1}{k}$

El agente aleatorio:

- no mantiene memoria;
- no realiza planificación;
- no utiliza conocimiento sobre la posición de la meta.

In [9]:
# -----------------------------
# Agente Aleatorio
# -----------------------------
class RandomAgent:
    def __init__(self, env):
        self.env = env

    def act(self):
        actions = self.env.perceive()
        action = random.choice(actions)
        self.env.step(action)

## Prueba del agente aleatorio

Ejecuta varias veces la siguiente celda.

Observa que el agente puede seguir trayectorias diferentes debido a que la selección de acciones es aleatoria.

In [14]:
env_random = GridWorld(
    size=3,
    start=(0, 0),
    goal=(2, 2)
)

random_agent = RandomAgent(env_random)

success, steps, trajectory = run_episode(
    env_random,
    random_agent,
    max_steps=20
)

print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Éxito: True
Pasos: 14
Trayectoria: [(0, 0), (0, 1), (0, 2), (0, 1), (1, 1), (2, 1), (2, 0), (2, 1), (1, 1), (2, 1), (1, 1), (1, 0), (1, 1), (1, 2), (2, 2)]


## Evaluación experimental

Una sola ejecución no es suficiente para comparar agentes, especialmente cuando alguno de ellos tiene comportamiento aleatorio.

Ejecutaremos cada agente durante múltiples episodios y calcularemos dos métricas.

### Tasa de éxito

La tasa de éxito representa la proporción de episodios en los que el agente alcanza la meta:

$\text{Tasa de éxito} =
\frac{\text{episodios exitosos}}
{\text{episodios totales}}$

### Número promedio de pasos

Para los episodios exitosos calcularemos:

$\overline{L} =
\frac{1}{N_s}
\sum_{i=1}^{N_s} L_i$

donde:

- $N_s$ es el número de episodios exitosos;
- $L_i$ es el número de pasos utilizado en el episodio $i$.

Un buen agente debería alcanzar la meta con alta frecuencia y utilizar pocos pasos.

In [15]:
def evaluate_agent(agent_type, episodes=100, max_steps=20):
    successes = 0
    successful_steps = []

    for _ in range(episodes):

        env = GridWorld(
            size=3,
            start=(0, 0),
            goal=(2, 2)
        )

        if agent_type == "reactive":
            agent = SimpleReactiveAgent(env, policy)

        elif agent_type == "random":
            agent = RandomAgent(env)

        else:
            raise ValueError("Tipo de agente no válido")

        success, steps, _ = run_episode(
            env,
            agent,
            max_steps
        )

        if success:
            successes += 1
            successful_steps.append(steps)

    success_rate = successes / episodes

    if successful_steps:
        avg_steps = sum(successful_steps) / len(successful_steps)
    else:
        avg_steps = None

    return success_rate, avg_steps

## Comparación de los agentes

Ejecutaremos ambos agentes bajo las mismas condiciones:

- mismo tamaño del entorno;
- mismo estado inicial;
- misma meta;
- mismo número máximo de pasos.

El agente reactivo es determinista, por lo que esperamos que repita el mismo comportamiento en todos los episodios.

El agente aleatorio puede producir resultados distintos entre ejecuciones.

In [18]:
reactive_success, reactive_steps = evaluate_agent(
    "reactive",
    episodes=100,
    max_steps=20
)

random_success, random_steps = evaluate_agent(
    "random",
    episodes=100,
    max_steps=20
)

print("Agente reactivo")
print("Tasa de éxito:", reactive_success)
print("Pasos promedio:", reactive_steps)

print("\nAgente aleatorio")
print("Tasa de éxito:", random_success)
print("Pasos promedio:", random_steps)

Agente reactivo
Tasa de éxito: 1.0
Pasos promedio: 4.0

Agente aleatorio
Tasa de éxito: 0.67
Pasos promedio: 10.477611940298507


In [19]:
print(f"{'Agente':<15} {'Éxito':<12} {'Pasos promedio'}")
print("-" * 45)

print(
    f"{'Reactivo':<15} "
    f"{reactive_success:<12.2f} "
    f"{reactive_steps:.2f}"
)

print(
    f"{'Aleatorio':<15} "
    f"{random_success:<12.2f} "
    f"{random_steps:.2f}"
)

Agente          Éxito        Pasos promedio
---------------------------------------------
Reactivo        1.00         4.00
Aleatorio       0.67         10.48


## Preguntas de reflexión

1. ¿Por qué el agente reactivo obtiene siempre el mismo resultado?

    Porque la solución la tiene codificada de manera explícita en su política, no es variable y siempre tomará las mismas acciones dada una     cierta posición.

2. ¿Por qué el agente aleatorio puede producir trayectorias diferentes?

    Porque no tiene una política definida de manera explícita, sus decisiones son aleatorias y por ende puede producir cualquier ruta           dentro de las posibilidades del espacio, incluso si estas no llevan a la meta.

3. ¿Alguno de los dos agentes realiza búsqueda en el espacio de estados?

    No, ningúno realiza una búsqueda, el reactivo simple solamente actúa como se le dice que debe hacerlo de acuerdo a su estado actual y       el aleatorio simplemente elige al azar de la lista de posibles acciones.

4. ¿Por qué el agente reactivo puede encontrar una trayectoria óptima sin utilizar un algoritmo de búsqueda?

    Porque nosotros ya codificamos esa trayectoria óptima en su política, si en lugar de esa política hicieramos una en la que el agente        toma más pasos de los que son necesarios este tomaría esa trayectoria. Es totalmente dependiente de lo que decidimos que debe hacer.

5. ¿Qué conocimiento sobre el problema está implícitamente codificado en la política tabular?

    La meta y la ruta más óptima para llegar a esta.

6. ¿Qué ocurriría si cambiamos la posición de la meta pero mantenemos la misma política?

    El agente no llegaría a la meta, llegaría al mismo lugar que tiene definido actualmente o se encontraría con la meta en su camino.

7. ¿La política tabular utilizada puede considerarse general para cualquier entorno `GridWorld`?

    No, cualquier entorno de tamaño distinto con una meta en otra posición no podría ser resuelto con la política actual, se tendría que        cambiar para cada uno de los posibles entornos.

8. ¿Qué ventaja ofrece el agente reactivo frente al agente aleatorio?

    Su principal ventaja es la eficiencia con la que llega a la meta y la certeza que tiene en lograrlo. Si se conoce la meta y el camino       para llegar a esta siempre obtendrá el mejor resultado posible. El aleatorio en promedio obtendrá una peor solución y en algunos casos      ni siquiera llegará a la meta.

9. ¿Qué limitación importante presenta una política tabular cuando aumenta el número de estados?

    Que para cada estado se tiene que codificar su acción respectiva, mientras más estados existan se necesita más memoria para almacenar       su respectiva acción y trabajo para definir las mismas.

## Reto

Genera de forma pseudoaleatoria un estado inicial y una meta distintos dentro del `GridWorld`.

Después, diseña una política reactiva que permita al agente alcanzar siempre la meta desde cualquier estado inicial.

### Condiciones

- El estado inicial y la meta deben ser diferentes.
- No debes construir manualmente una tabla para cada combinación posible.
- La acción debe depender únicamente del estado actual y de la meta.

### Preguntas

1. ¿Qué reglas utilizaste para decidir el movimiento?
2. ¿El agente realiza búsqueda o planificación?
3. ¿Por qué esta política funciona para cualquier par inicio--meta en este entorno?

In [28]:
# Generar inicio y meta pseudoaleatorios

size = 10

start = (
    random.randrange(size),
    random.randrange(size)
)

goal = start

while goal == start:
    goal = (
        random.randrange(size),
        random.randrange(size)
    )

print("Inicio:", start)
print("Meta:", goal)

Inicio: (7, 0)
Meta: (8, 3)


In [29]:
class GeneralReactiveAgent:
    def __init__(self, env):
        self.env = env

    def act(self):
        x, y = self.env.agent_pos
        gx, gy = self.env.goal

        if x != gx or y != gy:
            if x < gx:
                action = "RIGHT"
            elif x > gx:
                action = "LEFT"
            if y < gy:
                action = "UP"
            elif y > gy: 
                action = "DOWN"
            self.env.step(action)
        return None

In [30]:
env_general = GridWorld(
    size=size,
    start=start,
    goal=goal
)

general_agent = GeneralReactiveAgent(env_general)

success, steps, trajectory = run_episode(
    env_general,
    general_agent,
    max_steps=20
)

print("Inicio:", start)
print("Meta:", goal)
print("Éxito:", success)
print("Pasos:", steps)
print("Trayectoria:", trajectory)

Inicio: (7, 0)
Meta: (8, 3)
Éxito: True
Pasos: 4
Trayectoria: [(7, 0), (7, 1), (7, 2), (7, 3), (8, 3)]
